<a id="top"></a>

# Lab L0806: write the description, design the schema

```yaml
title: "Lab L0806: write the description, design the schema"
keywords: tool description, rich description, parameter schema, enum, Literal, per-field description, required, derived schema, StructuredTool, pydantic, lab
estimated duration: 30
```

> **Lesson:** L08. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L08/objectives.md). Solutions: `L0806_lab_solutions.ipynb`. Pairs with the [L0805 demo](L0805_lecture.ipynb).
>
> **Pure Python — no API key needed.** You *design the tool as a typed function*; `bind_tools` would derive its schema and send it to the model — but building the tool, inspecting the derived schema, and validating arguments against it are all offline (no model call). (The teacher demos make the live calls; these labs make the design choices.)
>
> **After this lab you can:** write a docstring description *for the model*, design a tight schema by **typing the function** (required params, a `Literal` enum, per-field descriptions), read the JSON-Schema LangChain derives from it, and validate sample arguments against that derived schema — all in pure Python.

## Contents

- [1. Setup — a sparse tool to improve](#1-setup--a-sparse-tool-to-improve)
- [2. Problem 1 — Rewrite the description for the model](#2-problem-1--rewrite-the-description-for-the-model)
- [3. Problem 2 — Tighten the parameter schema](#3-problem-2--tighten-the-parameter-schema)
- [4. Problem 3 — Validate sample arguments against your schema](#4-problem-3--validate-sample-arguments-against-your-schema)
- [5. Problem 5 — Why an enum over a free string? (written)](#5-problem-5--why-an-enum-over-a-free-string-written)
- [6. Self-check](#6-self-check)

## 1. Setup — a sparse tool to improve

A `set_priority(ticket_id, level)` tool with a **sparse** docstring and a **loose** signature (a free-string `level` with a default, so it isn't even required). Print the schema LangChain derives from it — that's the starting point you'll tighten across the lab.

In [ ]:
import json
from typing import Literal

from langchain_core.tools import StructuredTool
from langchain_core.utils.function_calling import convert_to_openai_tool
from pydantic import ValidationError


# A sparse, loose starting point: a one-line docstring (no when / what / return
# guidance) and a free-string `level` with a default (so it isn't even required).
def set_priority_sparse(ticket_id: str, level: str = "low") -> dict[str, object]:
    """Sets priority."""
    return {"ticket_id": ticket_id, "level": level, "updated": True}


SPARSE_TOOL = StructuredTool.from_function(func=set_priority_sparse, name="set_priority")
print("sparse tool's derived schema (loose — any string level, level not required):")
print(json.dumps(convert_to_openai_tool(SPARSE_TOOL)["function"], indent=2))

[↑ Back to top](#top)

## 2. Problem 1 — Rewrite the description for the model

Write `RICH_DESCRIPTION`: a 3–5 sentence string that answers all four questions from the lecture — *what it does*, *when to call it*, *when NOT to call it*, *what it returns*. Include the accepted `level` values and a one-line example. This is the text that becomes the tool's **docstring** in Problem 2 (LangChain sends it to the model as the description). The check below asserts your description mentions the key facts; the *wording* is yours.

In [ ]:
RICH_DESCRIPTION = ""  # TODO: 3-5 sentences, written for the model's selection step

# Lightweight check that the description covers the essentials (not graded on wording):
low = RICH_DESCRIPTION.lower()
for needle in ["ticket", "low", "high", "return"]:
    print(f"mentions {needle!r}:", needle in low)
assert len(RICH_DESCRIPTION.split()) >= 25, "aim for 3-5 real sentences"

[↑ Back to top](#top)

## 3. Problem 2 — Tighten the parameter schema

Tighten the schema by **typing the function**. Write `set_priority` with `ticket_id: str` and `level: Literal["low", "medium", "high"]` (both required — no defaults), and a Google-style docstring whose **Args** give a per-field description (each with an example or the allowed values, per the lecture's rule of thumb). Build `TIGHT_TOOL` with `parse_docstring=True` and print the schema LangChain **derives** — required list, `level` enum, and a description on each field, none of it hand-written JSON.

In [ ]:
# TODO: write set_priority(ticket_id: str, level: Literal["low","medium","high"]) with a
#       Google-style docstring (summary + Args: one line per field). Both params required.
def set_priority(ticket_id: str, level: Literal["low", "medium", "high"]) -> dict[str, object]:
    """TODO: your description here.

    Args:
        ticket_id: TODO.
        level: TODO.
    """
    return {"ticket_id": ticket_id, "level": level, "updated": True}


# TODO: build TIGHT_TOOL with StructuredTool.from_function(..., parse_docstring=True),
#       then print convert_to_openai_tool(TIGHT_TOOL)["function"]["parameters"].
raise NotImplementedError("build TIGHT_TOOL and print its derived schema, then delete this line")

[↑ Back to top](#top)

## 4. Problem 3 — Validate sample arguments against your schema

The derived schema isn't just documentation — `TIGHT_TOOL.args_schema` is a **Pydantic model**, the same contract the model sees. Implement `validate(args)` that calls `TIGHT_TOOL.args_schema.model_validate(args)` and returns `"ok"`, or a short reason pulled from the raised `ValidationError`. Run it over the sample calls — one good, three bad.

In [ ]:
SAMPLES = [
    {"ticket_id": "T-42", "level": "high"},  # ok
    {"ticket_id": "T-7"},  # missing required level
    {"ticket_id": "T-9", "level": "urgent"},  # not in enum
    {"ticket_id": 9, "level": "low"},  # wrong type for ticket_id
]


def validate(args: dict[str, object]) -> str:
    # TODO: try TIGHT_TOOL.args_schema.model_validate(args) -> return "ok";
    #       except ValidationError as exc: pull exc.errors()[0] and return a short reason.
    raise NotImplementedError("your code here")


for s in SAMPLES:
    print(s, "->", validate(s))

[↑ Back to top](#top)

## 5. Problem 5 — Why an enum over a free string? (written)

Your tight schema makes `level` an enum instead of a free-form string. In a sentence or two: what does the enum buy you in terms of *model behavior* and *validation* (think about the L0805 demo's sparse-vs-rich contrast)?

_Write your answer by editing this cell (double-click)._

_TODO_

[↑ Back to top](#top)

## 6. Self-check

Compare against `L0806_lab_solutions.ipynb`. Done when your `RICH_DESCRIPTION` answers all four questions, the typed `set_priority` derives a schema with both fields required and a `level` enum, and `validate` returns `ok` for the first sample and a useful reason for the other three.

[↑ Back to top](#top)